In [5]:
# ============================================
# IMPORTS
# ============================================

# Import necessary libraries for data processing, modeling, and evaluation
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# ============================================
# DATA LOADING & PREPROCESSING
# ============================================

numeric_cols = ["Marketing_Spend", "R&D_Spend", "Administration_Costs", "Number_of_Employees"] # Define numerical feature columns used for training
def load_and_preprocess(file_path):
    # Load dataset from CSV file
    df = pd.read_csv(file_path)
    # Define preprocessing steps for numerical and categorical features
    preprocessor = ColumnTransformer([
    ("num", Pipeline([('imputer', SimpleImputer(strategy='mean')), # Handle missing numerical values
    ('scaler', StandardScaler())]), numeric_cols), # Normalize numerical features
    ("cat", Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), # Handle missing categorical values
    ('encoder', OneHotEncoder(handle_unknown="ignore"))]), ["Region"])]) # Convert categorical to numerical
    X = df[['Marketing_Spend', 'R&D_Spend', 'Administration_Costs', 'Number_of_Employees', 'Region']]
    y = df['Revenue']
    return X, y, preprocessor

# ============================================
# MODEL TRAINING
# ============================================

def train_model(X_train, y_train, preprocessor):
    # Create a pipeline combining preprocessing and regression model
    model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
    ])
    model.fit(X_train, y_train)
    # Train the model using the training data
    return model

# ============================================
# MODEL EVALUATION
# ============================================

def evaluate_model(model, X_test, y_test):
    # Generate predictions on test data
    y_pred = model.predict(X_test)
    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred) # Mean Absolute Error (average prediction error)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Root Mean Squared Error (penalizes large errors)
    r2 = r2_score(y_test, y_pred) # R-squared score (explains variance in data)
    print(f'MAE: {mae:.2f}\nRMSE: {rmse:.2f}\nR2: {r2:.2f}')

# ============================================
# PREDICTION FUNCTION
# ============================================

def predict_revenue(model, user_input, feature_columns):
    # Convert user input into DataFrame with correct column structure
    input_df = pd.DataFrame([user_input], columns=list(feature_columns)) # ensure column order matches
    pred = model.predict(input_df) # Make prediction using trained model
    print(f'Predicted Revenue: {pred[0]:.2f}')

# ============================================
# MAIN EXECUTION
# ============================================

def main():
     # Load dataset and preprocess features
     X, y, preprocessor = load_and_preprocess('788438_data.csv')
     
     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Split data into training and testing sets
     model = train_model(X_train, y_train, preprocessor) # Train the model
     evaluate_model(model, X_test, y_test) # Evaluate model performance

     # Interactive loop for user predictions
     import sys
     if sys.stdin.isatty():
        while True:
            try:
                marketing = float(input('Marketing Spend: '))
                rnd = float(input('R&D Spend: '))
                admin = float(input('Administration Costs: '))
                employees = int(input('Number of Employees: '))
                region = input('Region: ')
                allowed_regions = ["North America", "Europe", "Asia"]
                if region not in allowed_regions:
                    print("Invalid region. Please choose from: North America, Europe, Asia.")
                    continue
                user_input = {
                'Marketing_Spend': marketing,
                'R&D_Spend': rnd,
                'Administration_Costs': admin,
                'Number_of_Employees': employees,
                'Region': region
                }

                predict_revenue(model, user_input, X.columns)

                cont = input('Do you want to continue? (y/n) ')
                if cont.lower() != 'y':
                    break
            except ValueError:
                print('Invalid input. Please enter a valid number.')
            except KeyboardInterrupt:
                print('\nExiting by user request.')
                break

# Run the main function when script is executed
if __name__ == "__main__":             
    main()


MAE: 6648.40
RMSE: 8363.06
R2: 0.93
